# PyTorch Basics for Protein Machine Learning

This notebook covers essential PyTorch concepts and applies them to protein property prediction.

**Learning Objectives:**
- Work with PyTorch tensors and automatic differentiation
- Build neural networks using nn.Module
- Train a protein solubility classifier from scratch

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/)

In [ ]:
# Install dependencies (uncomment for Colab)
# !pip install torch numpy pandas scikit-learn matplotlib seaborn

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch version: {torch.__version__}")
print(f"Using device: {device}")

## Part 1: Tensor Basics

In [ ]:
# Creating tensors
print("=== Tensor Creation ===")

# From scratch
zeros = torch.zeros(3, 4)
ones = torch.ones(3, 4)
randn = torch.randn(3, 4)  # Standard normal
arange = torch.arange(0, 10, 2)  # Like np.arange

print(f"zeros shape: {zeros.shape}")
print(f"ones dtype: {ones.dtype}")
print(f"arange: {arange}")

# From Python/NumPy
from_list = torch.tensor([1.0, 2.0, 3.0])
from_numpy = torch.from_numpy(np.array([1, 2, 3]))
print(f"\nfrom_list: {from_list}")
print(f"from_numpy: {from_numpy}")

In [ ]:
# Tensor operations
print("=== Basic Operations ===")

a = torch.randn(3, 4)
b = torch.randn(3, 4)

# Element-wise
print(f"a + b shape: {(a + b).shape}")
print(f"a * b shape: {(a * b).shape}")

# Matrix multiplication
c = a @ b.T  # (3, 4) @ (4, 3) = (3, 3)
print(f"a @ b.T shape: {c.shape}")

# Reductions
print(f"\na.sum(): {a.sum():.4f}")
print(f"a.mean(dim=1): {a.mean(dim=1)}")
print(f"a.max(dim=0): values={a.max(dim=0).values}")

In [ ]:
# Broadcasting (same rules as NumPy)
print("=== Broadcasting ===")

# (3, 4) + (4,) -> (3, 4)
x = torch.randn(3, 4)
bias = torch.randn(4)
result = x + bias
print(f"(3, 4) + (4,) = {result.shape}")

# (3, 4) + (3, 1) -> (3, 4)
scale = torch.randn(3, 1)
result = x + scale
print(f"(3, 4) + (3, 1) = {result.shape}")

# Common pattern: batch normalization-like operation
batch = torch.randn(32, 64)  # batch_size=32, features=64
mean = batch.mean(dim=0)     # (64,)
std = batch.std(dim=0)       # (64,)
normalized = (batch - mean) / (std + 1e-8)
print(f"\nNormalized shape: {normalized.shape}")

In [ ]:
# Reshaping tensors
print("=== Reshaping ===")

x = torch.randn(2, 3, 4)
print(f"Original shape: {x.shape}")

# view/reshape
print(f"view(6, 4): {x.view(6, 4).shape}")
print(f"reshape(-1): {x.reshape(-1).shape}")  # Flatten

# transpose
print(f"transpose(0, 2): {x.transpose(0, 2).shape}")

# unsqueeze/squeeze
y = torch.randn(5)
print(f"\nOriginal: {y.shape}")
print(f"unsqueeze(0): {y.unsqueeze(0).shape}")  # Add batch dim
print(f"unsqueeze(-1): {y.unsqueeze(-1).shape}")  # Add last dim

z = torch.randn(1, 5, 1)
print(f"\nOriginal: {z.shape}")
print(f"squeeze(): {z.squeeze().shape}")  # Remove all size-1 dims

## Part 2: Automatic Differentiation (Autograd)

In [ ]:
# Basic gradient computation
print("=== Autograd Basics ===")

# Create tensor with gradient tracking
x = torch.tensor([2.0, 3.0], requires_grad=True)
print(f"x = {x}")
print(f"requires_grad: {x.requires_grad}")

# Forward pass: y = x^2 + 3x
y = x ** 2 + 3 * x
print(f"\ny = x^2 + 3x = {y}")

# We need a scalar for backward
loss = y.sum()
print(f"loss = y.sum() = {loss}")

# Backward pass
loss.backward()

# Gradient: d(x^2 + 3x)/dx = 2x + 3
print(f"\ndy/dx = 2x + 3 = {x.grad}")
print(f"Expected: {2 * x.detach() + 3}")

In [ ]:
# Linear regression example
print("=== Linear Regression from Scratch ===")

# Generate data: y = 2x + 1 + noise
X = torch.randn(100, 1)
y_true = 2 * X + 1 + 0.1 * torch.randn(100, 1)

# Parameters to learn
w = torch.randn(1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)

# Training loop
learning_rate = 0.1
losses = []

for epoch in range(100):
    # Forward pass
    y_pred = X * w + b
    loss = ((y_pred - y_true) ** 2).mean()
    losses.append(loss.item())
    
    # Backward pass
    loss.backward()
    
    # Update parameters (without gradient tracking)
    with torch.no_grad():
        w -= learning_rate * w.grad
        b -= learning_rate * b.grad
    
    # Clear gradients for next iteration
    w.grad.zero_()
    b.grad.zero_()

print(f"Learned: w = {w.item():.4f}, b = {b.item():.4f}")
print(f"True:    w = 2.0000, b = 1.0000")

# Plot loss curve
plt.figure(figsize=(8, 4))
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Training Loss')
plt.yscale('log')
plt.grid(True, alpha=0.3)
plt.show()

## Part 3: Building Neural Networks with nn.Module

In [ ]:
# Simple MLP
class SimpleMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(0.1)
    
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.dropout(x)
        x = self.fc3(x)
        return x

# Create model
model = SimpleMLP(input_dim=20, hidden_dim=64, output_dim=2)
print(model)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,}")

In [ ]:
# Test forward pass
x = torch.randn(32, 20)  # Batch of 32, input dim 20
output = model(x)
print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")

# Get probabilities
probs = F.softmax(output, dim=-1)
print(f"Probabilities sum: {probs.sum(dim=-1)[:5]}")

In [ ]:
# Inspect parameters
print("=== Model Parameters ===")
for name, param in model.named_parameters():
    print(f"{name:15s} | shape: {str(list(param.shape)):15s} | params: {param.numel():,}")

## Part 4: Protein Solubility Prediction

We'll build a classifier to predict protein solubility from sequence.

In [ ]:
# Generate synthetic protein dataset
# In practice, you would load real data from DeepSol, etc.

AMINO_ACIDS = 'ACDEFGHIKLMNPQRSTVWY'
AA_TO_IDX = {aa: i for i, aa in enumerate(AMINO_ACIDS)}

def generate_random_sequence(length):
    """Generate a random protein sequence."""
    return ''.join(np.random.choice(list(AMINO_ACIDS), size=length))

def compute_hydrophobicity(seq):
    """Simple hydrophobicity score."""
    hydrophobic = set('AVILMFYW')
    return sum(1 for aa in seq if aa in hydrophobic) / len(seq)

def compute_charge(seq):
    """Net charge at pH 7."""
    positive = seq.count('K') + seq.count('R')
    negative = seq.count('D') + seq.count('E')
    return (positive - negative) / len(seq)

# Generate dataset
n_samples = 2000
sequences = []
labels = []

for _ in range(n_samples):
    length = np.random.randint(50, 300)
    seq = generate_random_sequence(length)
    
    # Solubility heuristic: lower hydrophobicity + higher charge = more soluble
    hydro = compute_hydrophobicity(seq)
    charge = abs(compute_charge(seq))
    
    prob_soluble = 1 / (1 + np.exp(5 * (hydro - 0.35) - 3 * charge))
    label = 1 if np.random.random() < prob_soluble else 0
    
    sequences.append(seq)
    labels.append(label)

df = pd.DataFrame({'sequence': sequences, 'label': labels})
print(f"Dataset size: {len(df)}")
print(f"\nClass distribution:")
print(df['label'].value_counts())

In [ ]:
# Analyze dataset
df['length'] = df['sequence'].str.len()
df['hydrophobicity'] = df['sequence'].apply(compute_hydrophobicity)
df['charge'] = df['sequence'].apply(compute_charge)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Length distribution
for label, name in [(0, 'Insoluble'), (1, 'Soluble')]:
    axes[0].hist(df[df['label'] == label]['length'], bins=30, alpha=0.6, label=name)
axes[0].set_xlabel('Sequence Length')
axes[0].set_ylabel('Count')
axes[0].legend()
axes[0].set_title('Length Distribution')

# Hydrophobicity distribution
for label, name in [(0, 'Insoluble'), (1, 'Soluble')]:
    axes[1].hist(df[df['label'] == label]['hydrophobicity'], bins=30, alpha=0.6, label=name)
axes[1].set_xlabel('Hydrophobicity')
axes[1].legend()
axes[1].set_title('Hydrophobicity Distribution')

# Charge distribution
for label, name in [(0, 'Insoluble'), (1, 'Soluble')]:
    axes[2].hist(df[df['label'] == label]['charge'], bins=30, alpha=0.6, label=name)
axes[2].set_xlabel('Net Charge')
axes[2].legend()
axes[2].set_title('Charge Distribution')

plt.tight_layout()
plt.show()

In [ ]:
# Create PyTorch Dataset
class ProteinDataset(Dataset):
    def __init__(self, sequences, labels, max_len=300):
        self.sequences = sequences
        self.labels = labels
        self.max_len = max_len
    
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        seq = self.sequences[idx]
        label = self.labels[idx]
        
        # Encode sequence (integer encoding)
        encoded = torch.zeros(self.max_len, dtype=torch.long)
        for i, aa in enumerate(seq[:self.max_len]):
            if aa in AA_TO_IDX:
                encoded[i] = AA_TO_IDX[aa] + 1  # 0 = padding
        
        # Create mask
        seq_len = min(len(seq), self.max_len)
        mask = torch.zeros(self.max_len, dtype=torch.float)
        mask[:seq_len] = 1.0
        
        return {
            'sequence': encoded,
            'mask': mask,
            'length': seq_len,
            'label': torch.tensor(label, dtype=torch.long)
        }

# Test dataset
test_ds = ProteinDataset(sequences[:5], labels[:5])
sample = test_ds[0]
print(f"Sample keys: {sample.keys()}")
print(f"Sequence shape: {sample['sequence'].shape}")
print(f"Mask shape: {sample['mask'].shape}")
print(f"Length: {sample['length']}")
print(f"Label: {sample['label']}")

In [ ]:
# Split data
train_df, temp_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['label'], random_state=42)

print(f"Train: {len(train_df)}")
print(f"Val: {len(val_df)}")
print(f"Test: {len(test_df)}")

# Create datasets
train_dataset = ProteinDataset(train_df['sequence'].tolist(), train_df['label'].tolist())
val_dataset = ProteinDataset(val_df['sequence'].tolist(), val_df['label'].tolist())
test_dataset = ProteinDataset(test_df['sequence'].tolist(), test_df['label'].tolist())

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64)
test_loader = DataLoader(test_dataset, batch_size=64)

In [ ]:
# Define model
class ProteinCNNClassifier(nn.Module):
    def __init__(self, vocab_size=21, embed_dim=64, hidden_dim=128, num_classes=2):
        super().__init__()
        
        # Embedding layer (0 = padding)
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        
        # Convolutional layers
        self.conv1 = nn.Conv1d(embed_dim, hidden_dim, kernel_size=5, padding=2)
        self.conv2 = nn.Conv1d(hidden_dim, hidden_dim, kernel_size=5, padding=2)
        self.conv3 = nn.Conv1d(hidden_dim, hidden_dim, kernel_size=5, padding=2)
        
        # Batch normalization
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.bn2 = nn.BatchNorm1d(hidden_dim)
        self.bn3 = nn.BatchNorm1d(hidden_dim)
        
        # Classifier
        self.fc = nn.Linear(hidden_dim, num_classes)
        self.dropout = nn.Dropout(0.3)
    
    def forward(self, x, mask=None):
        # x: (batch, seq_len) integer indices
        
        # Embed
        x = self.embedding(x)  # (batch, seq_len, embed_dim)
        x = x.transpose(1, 2)  # (batch, embed_dim, seq_len)
        
        # Convolutions
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.dropout(x)
        x = F.relu(self.bn2(self.conv2(x)))
        x = self.dropout(x)
        x = F.relu(self.bn3(self.conv3(x)))
        
        # Global average pooling with mask
        if mask is not None:
            mask = mask.unsqueeze(1)  # (batch, 1, seq_len)
            x = (x * mask).sum(dim=2) / mask.sum(dim=2).clamp(min=1)
        else:
            x = x.mean(dim=2)
        
        # Classify
        x = self.fc(x)
        return x

# Create model
model = ProteinCNNClassifier()
model = model.to(device)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Training functions
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for batch in loader:
        seq = batch['sequence'].to(device)
        mask = batch['mask'].to(device)
        labels = batch['label'].to(device)
        
        # Forward pass
        outputs = model(seq, mask)
        loss = criterion(outputs, labels)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        # Track metrics
        total_loss += loss.item()
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    
    return total_loss / len(loader), correct / total

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    
    for batch in loader:
        seq = batch['sequence'].to(device)
        mask = batch['mask'].to(device)
        labels = batch['label'].to(device)
        
        outputs = model(seq, mask)
        loss = criterion(outputs, labels)
        
        total_loss += loss.item()
        preds = outputs.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    
    accuracy = accuracy_score(all_labels, all_preds)
    return total_loss / len(loader), accuracy, all_preds, all_labels

In [ ]:
# Training loop
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=5, factor=0.5, verbose=True
)

n_epochs = 30
best_val_loss = float('inf')
patience = 10
patience_counter = 0

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

for epoch in range(n_epochs):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion, device)
    
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    scheduler.step(val_loss)
    
    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        best_model_state = model.state_dict().copy()
    else:
        patience_counter += 1
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:3d} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")
    
    if patience_counter >= patience:
        print(f"Early stopping at epoch {epoch+1}")
        break

# Load best model
model.load_state_dict(best_model_state)

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['val_loss'], label='Validation')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history['train_acc'], label='Train')
axes[1].plot(history['val_acc'], label='Validation')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Evaluate on test set
test_loss, test_acc, test_preds, test_labels = evaluate(model, test_loader, criterion, device)

print("=== Test Set Performance ===")
print(f"Loss: {test_loss:.4f}")
print(f"Accuracy: {test_acc:.4f}")
print("\nClassification Report:")
print(classification_report(test_labels, test_preds, target_names=['Insoluble', 'Soluble']))

In [ ]:
# Confusion matrix
cm = confusion_matrix(test_labels, test_preds)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Insoluble', 'Soluble'],
            yticklabels=['Insoluble', 'Soluble'])
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
# Make predictions on new sequences
@torch.no_grad()
def predict_solubility(model, sequences, device):
    model.eval()
    
    # Create a temporary dataset
    dataset = ProteinDataset(sequences, [0] * len(sequences))
    loader = DataLoader(dataset, batch_size=32)
    
    all_probs = []
    for batch in loader:
        seq = batch['sequence'].to(device)
        mask = batch['mask'].to(device)
        
        logits = model(seq, mask)
        probs = F.softmax(logits, dim=-1)
        all_probs.extend(probs[:, 1].cpu().numpy())  # Probability of soluble
    
    return np.array(all_probs)

# Example predictions
test_sequences = [
    "MKKKKKKKKKKEEEEEEEEEE",  # Charged, likely soluble
    "WWWWWWFFFFFFFF",         # Hydrophobic, likely insoluble
    "MKTAYIAKQRQISFVKSHFSRQLE",  # Mixed
]

probs = predict_solubility(model, test_sequences, device)

print("=== Predictions ===")
for seq, prob in zip(test_sequences, probs):
    label = "Soluble" if prob > 0.5 else "Insoluble"
    print(f"{seq[:30]:30s}... -> {label} (p={prob:.3f})")

## Summary

In this notebook, we covered:

1. **Tensor Basics**: Creating, manipulating, and operating on PyTorch tensors
2. **Autograd**: Automatic differentiation for computing gradients
3. **nn.Module**: Building neural network models
4. **Training Loop**: Complete workflow for training and evaluating models
5. **Protein Classification**: Applying these concepts to predict protein solubility

**Key takeaways:**
- PyTorch tensors are like NumPy arrays with GPU support and automatic differentiation
- The training loop consists of: forward pass -> loss -> backward pass -> optimizer step
- Always use `model.train()` for training and `model.eval()` for inference
- Early stopping and learning rate scheduling help prevent overfitting